# I. Tiền Xử Lý Dữ Liệu Spam SMS — Phiên bản nâng cấp

Pipeline gồm 9 bước: EDA → Xoá duplicate → Encode label → Làm sạch + Stemming → Tokenize + Stopwords → Train/Test Split → **Feature Engineering (trên train)** → Vector hoá (TF-IDF) → Balance Dataset (SMOTE)

> **Các cải tiến so với phiên bản cũ:**
> - ✅ **Sửa data leakage**: Feature selection (tương quan) được thực hiện **chỉ trên tập train**, không dùng toàn bộ dataset
> - ✅ **Thêm Stemming (PorterStemmer)**: Đưa các biến thể từ về gốc (running→run), giúp TF-IDF chính xác hơn
> - ✅ **Lưu `NUMERIC_FEATURES`**: Danh sách feature quan trọng được lưu ra file để đảm bảo train/predict nhất quán
> - ✅ **TF-IDF bigram**: Thêm `ngram_range=(1,2)` để bắt cụm từ spam như "click here", "act now"
> - ✅ **Chuẩn hoá numeric features**: Dùng StandardScaler để tránh một số feature có scale lớn lấn át TF-IDF


### Cài đặt thư viện

In [ ]:
!pip install pandas scikit-learn nltk matplotlib seaborn imbalanced-learn

### Import thư viện

In [ ]:
import pandas as pd
import numpy as np
import re
import pickle
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer          # ✅ MỚI: Stemming
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler   # ✅ MỚI: Chuẩn hoá numeric
from scipy.sparse import save_npz, hstack as sp_hstack, csr_matrix
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

nltk.download('stopwords')
print('Import xong!')


### Bước 0: Load dữ liệu gốc

In [ ]:
df = pd.read_csv('../data/spam.csv', encoding='latin-1')
df = df[['label', 'text']].rename(columns={'text': 'message'})
print(f'Shape: {df.shape}')
df.head(3)


### Bước 1: EDA — Khám phá dữ liệu

In [ ]:
print('Phân phối nhãn (giá trị gốc):')
print(df['label'].value_counts(dropna=False))
df['message_length'] = df['message'].astype(str).str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['#4C72B0','#DD8452'])
axes[0].set_title('Phân phối nhãn'); axes[0].set_xlabel('Nhãn'); axes[0].set_ylabel('Số lượng')

for lbl in df['label'].unique():
    sns.histplot(df[df['label']==lbl]['message_length'], label=str(lbl), alpha=0.6, bins=50, ax=axes[1])
axes[1].set_title('Phân phối độ dài tin nhắn'); axes[1].legend()

plt.tight_layout(); plt.show()
print(f'\nMissing values:\n{df.isnull().sum()}')
print(f'Số mẫu: {len(df)} | Độ dài TB: {df["message_length"].mean():.1f} ký tự')


### Bước 2: Xoá duplicate

In [ ]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after = len(df)
print(f'Trước: {before} | Sau: {after} | Đã xoá: {before-after} dòng trùng')


### Bước 3: Encode Label

In [ ]:
if df['label'].dtype == object:
    df['label'] = df['label'].astype(str).str.strip().map({'ham': 0, 'spam': 1})

if df['label'].isnull().any():
    print('Cảnh báo: còn label NaN!')

print('Phân phối nhãn sau encode:')
print(df['label'].value_counts(dropna=False))
print(f'Tỷ lệ spam: {df["label"].mean():.2%}')


### Bước 4: Làm sạch văn bản + Stemming

> **✅ Cải tiến:** Thêm **PorterStemmer** để đưa các biến thể từ về gốc.  
> Ví dụ: `winning`, `wins`, `won` → đều về `win`. Giúp TF-IDF không bỏ sót các từ spam biến thể.


In [ ]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text, flags=re.MULTILINE)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'\b\d{10,}\b', ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize_stem(text):
    """Tokenize, loại stopword, rồi stem."""
    tokens = str(text).split()
    tokens = [stemmer.stem(t) for t in tokens if t not in stop_words and len(t) > 1]
    return ' '.join(tokens)

df['clean_msg'] = df['message'].apply(clean_text)
df['processed'] = df['clean_msg'].apply(tokenize_stem)

print('Đã clean + stem xong!')
print(f'\nVí dụ gốc   : {df["message"].iloc[1][:80]}')
print(f'Sau clean   : {df["clean_msg"].iloc[1][:80]}')
print(f'Sau stem    : {df["processed"].iloc[1][:80]}')


### Bước 5: Tính Numeric Features (toàn bộ DataFrame)

In [ ]:
spam_keywords = ['free','win','prize','urgent','call','text','claim','cash','money','offer',
                 'winner','bonus','reward','congratulations','selected']
spam_tags     = ['act now','guaranteed','limited time','buy now','click here','urgent','risk free',
                 'you have won','claim your']
currency_syms = ['$','£','€','¥']

def build_numeric_features(df_in):
    d = df_in.copy()
    msg = d['message'].astype(str)
    cln = d['clean_msg'].astype(str)
    d['word_count']           = cln.apply(lambda x: len(x.split()))
    d['uppercase_words']      = msg.apply(lambda x: len([w for w in x.split() if w.isupper() and len(w)>1]))
    d['uppercase_ratio']      = d['uppercase_words'] / (d['word_count'] + 1)
    d['special_chars']        = msg.apply(lambda x: len(re.findall(r'[^\w\s]', x)))
    d['digit_count']          = msg.apply(lambda x: len(re.findall(r'\d', x)))
    d['long_words']           = cln.apply(lambda x: len([w for w in x.split() if len(w)>6]))
    d['spam_keywords']        = cln.apply(lambda x: sum(1 for w in x.split() if w in spam_keywords))
    d['special_ratio']        = d['special_chars'] / (d['message_length'] + 1)
    d['sentence_count']       = msg.apply(lambda x: len(re.findall(r'[.!?]', x)))
    d['avg_word_length']      = cln.apply(lambda x: np.mean([len(w) for w in x.split()]) if x.split() else 0)
    d['exclamation_count']    = msg.apply(lambda x: x.count('!'))
    d['question_count']       = msg.apply(lambda x: x.count('?'))
    d['consecutive_punct']    = msg.apply(lambda x: len(re.findall(r'([.!?])\1{1,}', x)))
    d['currency_symbol_count']= msg.apply(lambda x: sum(x.count(s) for s in currency_syms))
    d['has_spam_tag']         = cln.apply(lambda x: int(any(tag in x for tag in spam_tags)))
    d['caps_ratio']           = msg.apply(lambda x: sum(1 for c in x if c.isupper()) / (len(x)+1))
    return d

df = build_numeric_features(df)

ALL_NUMERIC = [
    'message_length','word_count','uppercase_words','uppercase_ratio',
    'special_chars','digit_count','long_words','spam_keywords',
    'special_ratio','sentence_count','avg_word_length',
    'exclamation_count','question_count','consecutive_punct',
    'currency_symbol_count','has_spam_tag','caps_ratio'
]
print(f'Đã tạo {len(ALL_NUMERIC)} numeric features.')
print(df[ALL_NUMERIC].describe().round(2))


### Bước 6: Train / Test Split

> **⚠️ Quan trọng:** Chia train/test **TRƯỚC** khi chọn feature — tránh data leakage.


In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['processed'], df['label'],
    test_size=0.2, stratify=df['label'], random_state=42
)

# Tách numeric features theo index
X_train_num_all = df.loc[X_train_text.index, ALL_NUMERIC].values
X_test_num_all  = df.loc[X_test_text.index,  ALL_NUMERIC].values

print(f'Train: {len(X_train_text)} mẫu | Test: {len(X_test_text)} mẫu')
print('\nTỉ lệ lớp train:')
print(y_train.value_counts(normalize=True).round(3))


### Bước 7: Feature Selection (TRÊN TRAIN — tránh data leakage)

> **✅ Sửa lỗi cũ:** Trước đây feature selection dùng toàn bộ `df` (kể cả test) → data leakage.  
> Bây giờ chỉ tính tương quan trên **tập train**, chọn top 8.


In [ ]:
# Tính tương quan chỉ trên train
train_df_corr = pd.DataFrame(X_train_num_all, columns=ALL_NUMERIC)
train_df_corr['label'] = y_train.values

corr_train = train_df_corr.corr()['label'].abs().drop('label').sort_values(ascending=False)
NUMERIC_FEATURES = corr_train.index.tolist()[:8]

print('Top 8 numeric features (chọn từ train):')
for i, (feat, val) in enumerate(corr_train.head(8).items(), 1):
    print(f'  {i}. {feat:<25} corr={val:.4f}')

# Lưu danh sách feature để dùng khi predict
with open('../data/numeric_features.pkl', 'wb') as f:
    pickle.dump(NUMERIC_FEATURES, f)
print('\n✅ Đã lưu NUMERIC_FEATURES vào ../data/numeric_features.pkl')


### Bước 7b: Chuẩn hoá Numeric Features

In [ ]:
# Chỉ lấy 8 features đã chọn
X_train_num = X_train_num_all[:, [ALL_NUMERIC.index(f) for f in NUMERIC_FEATURES]]
X_test_num  = X_test_num_all[:,  [ALL_NUMERIC.index(f) for f in NUMERIC_FEATURES]]

# Fit scaler chỉ trên train
scaler = StandardScaler()
X_train_num_scaled = scaler.fit_transform(X_train_num)
X_test_num_scaled  = scaler.transform(X_test_num)

# Lưu scaler
with open('../data/numeric_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('Numeric features (scaled):')
print(f'  X_train_num: {X_train_num_scaled.shape}')
print(f'  X_test_num : {X_test_num_scaled.shape}')


### Bước 8: Vector Hoá TF-IDF

> **✅ Cải tiến:** Thêm `ngram_range=(1,2)` để bắt cụm từ spam ("free money", "click here").


In [ ]:
vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),      # ✅ Bigram — bắt cụm từ
    sublinear_tf=True,       # ✅ Log TF — giảm ảnh hưởng từ lặp nhiều lần
    min_df=2                 # ✅ Bỏ từ xuất hiện < 2 lần
)

X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_test_tfidf  = vectorizer.transform(X_test_text)

print(f'TF-IDF train: {X_train_tfidf.shape}')
print(f'TF-IDF test : {X_test_tfidf.shape}')

# Kết hợp TF-IDF + numeric (scaled)
X_train_combined = sp_hstack([X_train_tfidf, csr_matrix(X_train_num_scaled)])
X_test_combined  = sp_hstack([X_test_tfidf,  csr_matrix(X_test_num_scaled)])

print(f'\nCombined train: {X_train_combined.shape}')
print(f'Combined test : {X_test_combined.shape}')


### Bước 9: Balance Dataset (SMOTE)

> **Lưu ý:** SMOTE chỉ trên **train**, test giữ nguyên.


In [ ]:
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_combined, y_train)

print(f'Trước SMOTE: {X_train_combined.shape} | Phân phối: {dict(y_train.value_counts().sort_index())}')
counts = pd.Series(y_train_balanced).value_counts().sort_index()
print(f'Sau  SMOTE: {X_train_balanced.shape}  | Ham(0): {counts[0]} | Spam(1): {counts[1]}')


### Lưu kết quả

In [ ]:
import os
os.makedirs('../data', exist_ok=True)

df.to_csv('../data/spam_processed.csv', index=False)

train_df = pd.DataFrame({'text': X_train_text, 'label': y_train})
test_df  = pd.DataFrame({'text': X_test_text,  'label': y_test})
train_df.to_csv('../data/train.csv', index=False)
test_df.to_csv('../data/test.csv',   index=False)

save_npz('../data/X_train.npz', X_train_balanced)
save_npz('../data/X_test.npz',  X_test_combined)
np.save('../data/y_train.npy',  y_train_balanced)
np.save('../data/y_test.npy',   y_test.values)

with open('../data/tfidf_vectorizer.pkl', 'wb') as f: pickle.dump(vectorizer, f)
with open('../data/numeric_features.pkl', 'wb') as f: pickle.dump(NUMERIC_FEATURES, f)
with open('../data/numeric_scaler.pkl',   'wb') as f: pickle.dump(scaler, f)

print('✅ Đã lưu:')
print('  X_train.npz / X_test.npz     — TF-IDF + numeric combined')
print('  y_train.npy / y_test.npy     — nhãn')
print('  tfidf_vectorizer.pkl          — TF-IDF vectorizer')
print('  numeric_features.pkl          — danh sách 8 numeric features')
print('  numeric_scaler.pkl            — StandardScaler')
